# TD 9 — Corrigé

**Analyse des données — L3 Économie**

Mis en ligne après la séance.

> Les cours sont téléchargés en direct : les chiffres dépendent du jour d'exécution. Les **conclusions**, elles, ne changent pas.


## Partie 1 — Le piège : prévoir le prix

In [ ]:
!pip install -q yfinance

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import yfinance as yf

TITRE  = "RNO.PA"      # Renault
MARCHE = "^FCHI"       # CAC 40

brut = yf.download([TITRE, MARCHE], start="2019-01-01", end="2025-12-31",
                   auto_adjust=True, progress=False)

px  = brut["Close"].dropna()
vlm = brut["Volume"].dropna()
px.columns  = ["MARCHE" if c == MARCHE else "TITRE" for c in px.columns]
vlm.columns = ["MARCHE" if c == MARCHE else "TITRE" for c in vlm.columns]

print(px.shape, "| du", px.index.min().date(), "au", px.index.max().date())
px.tail(3).round(2)

In [ ]:
r = np.log(px).diff().dropna()
print(len(r), "seances")

In [ ]:
p = pd.DataFrame({"demain": px["TITRE"].shift(-1), "aujourdhui": px["TITRE"]}).dropna()
mod = smf.ols("demain ~ aujourdhui", data=p).fit()
print("R2 =", round(mod.rsquared, 5))
print("pente :", round(mod.params["aujourdhui"], 4))

In [ ]:
from sklearn.metrics import mean_squared_error

err_modele = np.sqrt(mean_squared_error(p["demain"], mod.fittedvalues))
err_naif   = np.sqrt(mean_squared_error(p["demain"], p["aujourdhui"]))

print("RMSE du modele estime        :", round(err_modele, 4), "EUR")
print("RMSE de 'demain = aujourdhui':", round(err_naif, 4), "EUR")
print("gain du modele               :", round(100 * (1 - err_modele / err_naif), 2), "%")

**Réponses 2 et 3.** Le $R^2$ dépasse 0,99 et la pente vaut pratiquement 1. Présenté seul, ce modèle a l'air extraordinaire.

Il ne vaut rien. L'erreur du modèle estimé est à peine inférieure à celle de la règle **« demain, le prix sera celui d'aujourd'hui »** — une règle sans paramètre, sans données, et qu'un enfant peut appliquer.

D'où vient le $R^2$ ? Un cours de bourse est une série très **persistante** : la variation d'un jour est minuscule devant le niveau. Expliquer 99 % de la variance d'un niveau, c'est seulement avoir remarqué que le prix de demain est proche de celui d'aujourd'hui. Le $R^2$ mesure ici la **persistance de la série**, pas la qualité de la prévision.

C'est pourquoi on travaille sur les **rendements** et non sur les prix : la bonne question n'est pas « où sera le cours ? » mais « de combien va-t-il bouger ? ».


## Partie 2 — Prévoir le rendement

In [ ]:
X = pd.DataFrame(index=px.index)

for k in range(1, 6):
    X[f"r_lag{k}"] = r["TITRE"].shift(k)          # les cinq derniers rendements

X["ma5"]     = r["TITRE"].shift(1).rolling(5).mean()
X["ma21"]    = r["TITRE"].shift(1).rolling(21).mean()
X["vol5"]    = r["TITRE"].shift(1).rolling(5).std()
X["vol21"]   = r["TITRE"].shift(1).rolling(21).std()
X["marche1"] = r["MARCHE"].shift(1)
X["lvolume"] = np.log(vlm["TITRE"]).shift(1)
X["lundi"]   = (X.index.dayofweek == 0).astype(int)

VARS = list(X.columns)
print(len(VARS), "variables :", VARS)

**Réponse 4.** Onze variables, toutes **décalées d'au moins un jour**. C'est la contrainte absolue de la prévision : à la clôture de la veille, on ne connaît que le passé.

Le `.shift(1)` avant chaque `.rolling()` n'est pas une précaution de style. Sans lui, la volatilité « des 5 derniers jours » inclut le jour qu'on cherche à prévoir — la question 13 montre ce que cela produit.


In [ ]:
def jeu(cible):
    """Assemble cible + variables, puis coupe 80 / 20 DANS LE TEMPS."""
    d = pd.concat([cible.rename("y"), X], axis=1).dropna()
    c = int(len(d) * 0.8)
    return d.iloc[:c], d.iloc[c:]

tr, te = jeu(r["TITRE"])
print("apprentissage :", tr.index[0].date(), "->", tr.index[-1].date(), f"  ({len(tr)})")
print("test          :", te.index[0].date(), "->", te.index[-1].date(), f"  ({len(te)})")

**Réponse 5.** Le découpage est **chronologique** : on apprend sur le passé, on teste sur l'avenir.

Découper au hasard reviendrait à entraîner le modèle sur des journées de 2025 pour prédire des journées de 2021. Aucun praticien ne dispose jamais de cette information, donc la performance mesurée serait imaginaire.

Il y a une seconde raison, propre aux séries temporelles : les variables construites sur fenêtres glissantes **se chevauchent**. Deux journées consécutives partagent presque toute leur moyenne mobile. En mélangeant, on place dans l'apprentissage des observations quasi identiques à celles du test — le modèle a déjà vu la réponse.


In [ ]:
def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def r2_hors(y_vrai, y_prev, repere):
    """1 - SCR du modele / SCR du repere. Peut etre negatif."""
    y_vrai = np.asarray(y_vrai)
    return 1 - np.sum((y_vrai - y_prev)**2) / np.sum((y_vrai - repere)**2)

In [ ]:
m = LinearRegression().fit(tr[VARS], tr["y"])
prev = m.predict(te[VARS])

print("R2 en APPRENTISSAGE :", round(m.score(tr[VARS], tr["y"]), 4))
print()
print("RMSE du modele          :", round(rmse(te["y"], prev), 6))
print("RMSE du repere 'zero'   :", round(rmse(te["y"], np.zeros(len(te))), 6))
print("RMSE du repere 'moyenne':", round(rmse(te["y"], np.full(len(te), tr["y"].mean())), 6))
print()
print("R2 HORS ECHANTILLON (vs zero)    :", round(r2_hors(te["y"], prev, 0), 4))
print("R2 HORS ECHANTILLON (vs moyenne) :",
      round(r2_hors(te["y"], prev, tr["y"].mean()), 4))

**Réponses 6 et 7.** Le $R^2$ en apprentissage est faible mais positif : avec onze variables sur plus de mille journées, il est **impossible** qu'il soit nul. Les moindres carrés exploitent le moindre hasard présent dans les données d'apprentissage.

Le $R^2$ hors échantillon, lui, est **proche de zéro et le plus souvent négatif**.

Un $R^2$ négatif n'est pas une anomalie de calcul. Il signifie que le modèle fait **pire que le repère** : on aurait mieux prédit en annonçant zéro tous les jours, sans rien estimer. Ce n'est possible que hors échantillon — en apprentissage, les moindres carrés garantissent de battre la moyenne. La différence entre les deux chiffres, c'est exactement le **surajustement** : le modèle a appris du bruit, et le bruit ne se reproduit pas.

Deux leçons de méthode.

Un modèle de prévision **n'existe pas sans son repère**. Annoncer « mon modèle obtient un $R^2$ de 0,01 » ne veut rien dire tant qu'on n'a pas dit par rapport à quoi.

Et le repère doit être **honnête** : ici, prédire zéro. Pas un repère volontairement mauvais choisi pour faire briller le modèle.


## Partie 3 — Prévoir seulement le sens

In [ ]:
hausse = (te["y"] > 0).mean()
juste  = ((prev > 0) == (te["y"] > 0)).mean()
n      = len(te)
marge  = 1.96 * 0.5 / np.sqrt(n)          # incertitude sur une proportion

print("seances en hausse dans le test :", round(100 * hausse, 1), "%")
print("  -> 'toujours en hausse' a raison", round(100 * hausse, 1), "% du temps")
print("le modele a raison             :", round(100 * juste, 1), "%")
print()
print(f"incertitude sur {n} journees : +/- {100*marge:.1f} points")

**Réponses 8 et 9.** Le **taux de base** est la part de séances en hausse — autour de la moitié, souvent un peu plus, puisque les marchés montent en moyenne sur longue période. Un « modèle » qui dirait *en hausse* tous les jours atteindrait ce score sans rien calculer.

Le modèle est en général très proche de ce taux de base, au-dessus ou en dessous selon la période. Et c'est là qu'il faut sortir la marge d'erreur : sur $n$ journées, l'incertitude sur une proportion est d'environ $\pm 1{,}96 \times 0{,}5/\sqrt{n}$, soit plusieurs points de pourcentage. Un modèle à 52 % contre un taux de base à 50 % n'a **rien démontré**.

C'est l'erreur la plus répandue dans les projets étudiants sur données financières : annoncer « 53 % de bonnes prédictions » comme une réussite, sans donner le taux de base ni l'incertitude. Les deux chiffres manquants sont ceux qui comptent.


## Partie 4 — Ce qui, lui, est prévisible

In [ ]:
tra, tea = jeu(r["TITRE"].abs())        # meme code, cible differente

ma = LinearRegression().fit(tra[VARS], tra["y"])
pa = ma.predict(tea[VARS])

print("cible = |rendement| (l'amplitude, sans le sens)")
print("  R2 en apprentissage      :", round(ma.score(tra[VARS], tra["y"]), 4))
print("  R2 HORS ECHANTILLON      :", round(r2_hors(tea["y"], pa, tra["y"].mean()), 4))

**Réponse 10.** Le $R^2$ hors échantillon devient **nettement positif**. Le même code, les mêmes variables, le même découpage — seule la cible a changé, et le résultat se renverse.

**Réponse 11.** Parce que les deux questions n'ont pas la même réponse dans le monde réel.

Le **sens** ne peut pas être prévisible. S'il l'était, il suffirait d'acheter chaque fois que le modèle annonce une hausse ; la demande ferait monter le prix immédiatement, et l'occasion disparaîtrait. Toute prévisibilité du sens est détruite par ceux qui l'exploitent. C'est le contenu concret de l'efficience des marchés — et c'est pourquoi un $R^2$ hors échantillon nul sur les rendements n'est pas un échec du modèle, mais une **confirmation** de la théorie.

L'**amplitude**, elle, n'offre aucune occasion de gain direct : savoir que demain sera agité ne dit pas dans quel sens. Rien ne pousse donc à l'arbitrer, et la régularité subsiste. C'est le **regroupement de volatilité** : les journées agitées se suivent, les calmes aussi.

Ce n'est pas une curiosité de salle de cours. Toute la gestion du risque repose dessus — valeur en risque, exigences de fonds propres, prix des options. On ne prévoit pas les rendements ; on prévoit leur dispersion, et cela suffit à faire un métier.

**Réponse 12.** Les trois phrases attendues :

> *Nous n'avons pas réussi à prévoir le rendement de demain : le $R^2$ hors échantillon est nul, voire négatif, et le modèle ne devine pas le sens mieux qu'une règle fixe.*
>
> *Nous avons réussi à prévoir son amplitude : le $R^2$ hors échantillon est nettement positif, parce que la volatilité se regroupe.*
>
> *Le premier résultat est le plus instructif : un échec de prévision bien mesuré est une information économique — il dit que le marché a déjà intégré ce que nous savons.*


## Pour ceux qui ont terminé

In [ ]:
# QUESTION 13 — l'oubli du .shift(1)
def essai(avec_shift):
    Z = X.copy()
    s = 1 if avec_shift else 0
    Z["vol5"] = r["TITRE"].shift(s).rolling(5).std()      # la seule ligne qui change
    d = pd.concat([r["TITRE"].abs().rename("y"), Z], axis=1).dropna()
    c = int(len(d) * .8)
    a, b = d.iloc[:c], d.iloc[c:]
    mm = LinearRegression().fit(a[VARS], a["y"])
    return r2_hors(b["y"], mm.predict(b[VARS]), a["y"].mean())

print("vol5 correcte, avec .shift(1) : R2 hors echantillon",
      round(essai(True), 3))
print("vol5 SANS .shift(1)           : R2 hors echantillon",
      round(essai(False), 3))

**Réponse 13.** Le $R^2$ hors échantillon bondit — il peut doubler.

D'où vient cette performance ? `rolling(5).std()` sans décalage calcule l'écart-type des cinq derniers jours **jour $t$ inclus**. Or la cible est $|r_t|$. La variable contient donc, pour un cinquième, la réponse elle-même. Le modèle ne prévoit pas : il **recopie**.

Cette fuite ne déclenche aucune erreur, aucun avertissement. Elle se voit à un seul signe : une performance trop belle. C'est la raison pour laquelle un résultat de prévision inattendu se vérifie toujours en remontant la construction de chaque variable, une par une, et en se demandant : *cette information était-elle disponible la veille au soir ?*

Le découpage chronologique ne protège pas de cette fuite-là. Les deux précautions sont indépendantes et toutes deux nécessaires.


In [ ]:
# QUESTION 14 — vingt variables de pur hasard, l'experience repetee 200 fois
rng = np.random.default_rng(0)
comptes, r2s = [], []

for essai_num in range(200):
    bruit = pd.DataFrame(rng.normal(size=(len(tr), 20)),
                         columns=[f"bruit{i}" for i in range(20)], index=tr.index)
    mb = smf.ols("y ~ " + " + ".join(bruit.columns),
                 data=pd.concat([tr["y"], bruit], axis=1)).fit()
    comptes.append(int((mb.pvalues.drop("Intercept") < 0.05).sum()))
    r2s.append(mb.rsquared)

comptes = np.array(comptes)
print("200 tirages de 20 variables de bruit pur :")
print("   nombre moyen de variables 'significatives' a 5 % :", round(comptes.mean(), 2))
print("   attendu par pur hasard                           : 1.0")
print("   au moins une significative dans", round(100 * (comptes >= 1).mean()), "% des tirages")
print("   trois ou plus dans", round(100 * (comptes >= 3).mean()), "% des tirages")
print()
print("   R2 moyen du modele de bruit pur :", round(float(np.mean(r2s)), 4))

**Réponse 14.** En moyenne **une sur vingt** ressort significative à 5 % — c'est la définition même du seuil : 5 % de chances de rejeter à tort. Un tirage isolé peut en donner zéro, deux ou quatre ; c'est la moyenne sur 200 tirages qui retombe sur 1.

Et dans une majorité de tirages, **au moins une** variable est significative. Avec vingt variables, trouver une étoile n'est donc pas un résultat : c'est le cas le plus fréquent, même quand aucune variable ne contient la moindre information. Le $R^2$, lui, n'est jamais nul.

La conséquence pratique est sévère : dès qu'on essaie beaucoup de variables, ou beaucoup de modèles, les $p$ affichés ne veulent plus dire ce qu'ils disent. C'est le *p-hacking*, et c'est précisément ce que la validation hors échantillon permet de détecter — le bruit appris ne se reproduit pas sur des données nouvelles.


In [ ]:
# QUESTION 15 — la meme chose avec une foret aleatoire
from sklearn.ensemble import RandomForestRegressor

for cible, nom in [(r["TITRE"], "rendement"), (r["TITRE"].abs(), "|rendement|")]:
    a, b = jeu(cible)
    lin = LinearRegression().fit(a[VARS], a["y"])
    rf  = RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                                random_state=0, n_jobs=-1).fit(a[VARS], a["y"])
    print(f"cible = {nom}")
    print("   lineaire : R2 apprentissage", round(lin.score(a[VARS], a["y"]), 3),
          "  hors echantillon", round(r2_hors(b["y"], lin.predict(b[VARS]), a["y"].mean()), 3))
    print("   foret    : R2 apprentissage", round(rf.score(a[VARS], a["y"]), 3),
          "  hors echantillon", round(r2_hors(b["y"], rf.predict(b[VARS]), a["y"].mean()), 3))

**Réponse 15.** À discuter en séance 10. Trois choses à observer dès maintenant.

La forêt obtient un $R^2$ d'**apprentissage** bien supérieur à celui de la régression : elle est beaucoup plus souple, donc elle épouse mieux les données qu'on lui montre.

Sur le **rendement**, cela ne lui sert à rien : hors échantillon, elle ne bat pas davantage le repère que la régression. Quand il n'y a rien à trouver, un modèle plus puissant ne trouve pas plus — il surajuste simplement davantage.

Sur l'**amplitude**, la comparaison devient intéressante, et c'est là qu'un modèle non linéaire peut apporter quelque chose.

La leçon à emporter en séance 10 : **ce n'est pas la puissance du modèle qui décide, c'est l'existence d'un signal.** Et le protocole — découpage chronologique, repère honnête, évaluation hors échantillon — ne change pas d'un modèle à l'autre. C'est lui qui permet de comparer.
